# 01 — Единая подготовка train / validation / test

Этот notebook теперь **показывает** подготовку данных, но не хранит её реализацию. Единственный источник правил — `src/ml_project/epic_data.py`. Поэтому новый эксперимент не копирует длинные функции и не может незаметно создать другое разбиение.

Цель notebook: пересобрать общий split, проверить его и объяснить интерфейс, который будут использовать все experiment-notebooks.


## Что где хранится

| Слой | Ответственность |
|---|---|
| `data/raw/unpacked/jaNemVect1.1/` | неизменяемые исходные файлы |
| `src/ml_project/epic_data.py` | BED-проверки, pooling r1+r2, split целых contig, manifest и потоковые batches |
| `data/processed/.../contig_holdout_v1/` | компактные интервалы, положительные targets, SHA-256 и состав split |
| `notebooks/experiments/EXP-*.ipynb` | только sampling, признаки, модель, метрики и интерпретация конкретной гипотезы |

Важно: официальный test остаётся нетронутым, и его ответы здесь не читаются.


In [9]:
from pathlib import Path
import sys

from IPython.display import display
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_data import DEFAULT_SPLIT_CONFIG, prepare_split

print("Project:", PROJECT_ROOT)
print("Data module:", SRC_DIR / "ml_project/epic_data.py")
print("Config:", DEFAULT_SPLIT_CONFIG)


Project: D:\Projects\EPIC\EPIC
Data module: D:\Projects\EPIC\EPIC\src\ml_project\epic_data.py
Config: SplitConfig(assembly='jaNemVect1.1', split_version='contig_holdout_v1', seed=42, validation_contig_fraction=0.25)


## Правило разбиения

1. Берём только contig официального train.
2. Сортируем их имена, затем перемешиваем `numpy.default_rng(seed)`.
3. Первые `ceil(25%)` contig становятся validation; остальные — train.
4. Обе цепи и обе csRNA-повторности одного contig всегда остаются вместе.
5. Официальные test-contig вообще не участвуют в выборе validation.

Seed 42 — не «магическое хорошее число», а просто зафиксированный идентификатор конкретного воспроизводимого разбиения. Менять его после просмотра результатов нельзя: это уже будет другая оценка.


In [10]:
# Пересобираем из raw, сравниваем с уже сохранённой версией и проверяем SHA-256.
split = prepare_split(PROJECT_ROOT)
print("Prepared:", split.paths.output)
print("Split version:", split.config.split_version)


Prepared: D:\Projects\EPIC\EPIC\data\processed\jaNemVect1.1\splits\contig_holdout_v1
Split version: contig_holdout_v1


## Фактический состав

`position_strand_count` считает каждую разрешённую координату отдельно для цепей `+` и `-`. В test столбцы с target намеренно пустые.


In [11]:
display(split.contig_assignment)
display(split.summary)


,contig,split,source_split
0,NC_064034.1,validation,train
1,NC_064035.1,train,train
2,NC_064036.1,train,train
3,NC_064038.1,train,train
4,NC_064039.1,train,train
5,NC_064040.1,train,train
6,NC_064041.1,validation,train
7,NC_064042.1,validation,train
8,NC_064043.1,train,train
9,NC_064045.1,train,train


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


## Почему не создаём таблицу на 384 миллиона строк

Whitelist компактно хранится интервалами. Для train/validation отдельно сохраняются только ненулевые pooled targets. Нулевые примеры восстанавливаются на лету небольшими batch. Так мы сохраняем **все** отрицательные позиции, но не расходуем десятки гигабайт RAM.


In [12]:
train_example = next(split.iter_batches("train", batch_size=8))
validation_example = next(split.iter_batches("validation", batch_size=8))
test_example = next(split.iter_batches("test", batch_size=8))

print("TRAIN: есть count и target")
display(train_example)
print("VALIDATION: есть count и target")
display(validation_example)
print("TEST: ответов нет")
display(test_example)


TRAIN: есть count и target


,source_split,template_index,contig,coordinate_0based,strand,split,count,target
0,train,16974557,NC_064035.1,2268,+,train,0,0
1,train,16974558,NC_064035.1,2269,+,train,0,0
2,train,16974559,NC_064035.1,2270,+,train,0,0
3,train,16974560,NC_064035.1,2271,+,train,0,0
4,train,16974561,NC_064035.1,2272,+,train,0,0
5,train,16974562,NC_064035.1,2273,+,train,0,0
6,train,16974563,NC_064035.1,2274,+,train,0,0
7,train,16974564,NC_064035.1,2275,+,train,0,0


VALIDATION: есть count и target


,source_split,template_index,contig,coordinate_0based,strand,split,count,target
0,train,0,NC_064034.1,1943,+,validation,0,0
1,train,1,NC_064034.1,1944,+,validation,0,0
2,train,2,NC_064034.1,1945,+,validation,0,0
3,train,3,NC_064034.1,1946,+,validation,0,0
4,train,4,NC_064034.1,1947,+,validation,0,0
5,train,5,NC_064034.1,1948,+,validation,0,0
6,train,6,NC_064034.1,1949,+,validation,0,0
7,train,7,NC_064034.1,1950,+,validation,0,0


TEST: ответов нет


,source_split,template_index,contig,coordinate_0based,strand,split
0,test,0,NC_064037.1,1814,+,test
1,test,1,NC_064037.1,1815,+,test
2,test,2,NC_064037.1,1816,+,test
3,test,3,NC_064037.1,1817,+,test
4,test,4,NC_064037.1,1818,+,test
5,test,5,NC_064037.1,1819,+,test
6,test,6,NC_064037.1,1820,+,test
7,test,7,NC_064037.1,1821,+,test


## Автоматические гарантии

При подготовке модуль проверяет границы BED, порядок и непересечение whitelist, точное соответствие template, уникальность position-strand, положение сигнала внутри whitelist, отсутствие общих contig между частями и полное покрытие исходных train/test. Manifest фиксирует SHA-256 исходников и каждого производного файла.


In [13]:
train_contigs = set(split.intervals("train")["contig"])
validation_contigs = set(split.intervals("validation")["contig"])
test_contigs = set(split.intervals("test")["contig"])

assert train_contigs.isdisjoint(validation_contigs)
assert train_contigs.isdisjoint(test_contigs)
assert validation_contigs.isdisjoint(test_contigs)
assert split.positive_targets("test") is None
assert "target" not in test_example

print("OK: contig не пересекаются, test labels отсутствуют, hashes проверены.")


OK: contig не пересекаются, test labels отсутствуют, hashes проверены.


## Как это использовать в новом эксперименте

В каждом новом notebook нужны только две строки:

```python
from ml_project.epic_data import ensure_prepared_split
split = ensure_prepared_split(PROJECT_ROOT)
```

Дальше эксперимент использует:

- `split.summary` — паспорт частей;
- `split.contig_assignment` — какой contig куда попал;
- `split.intervals("train")` — компактные разрешённые интервалы;
- `split.positive_targets("train")` — ненулевые pooled targets;
- `split.iter_batches("validation", batch_size=...)` — полные позиции, включая нули.

Sampling, признаки, модель и метрики **не входят** в общий модуль: именно они должны быть явно видны в notebook конкретного эксперимента.
